<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/microscopy-Core-ISMMS/ImageAnalysisCourse/blob/main/notebooks/03b_foundation_model_segmentation.ipynb)

*Click the badge to open this notebook in Google Colab. For best performance, switch to a GPU runtime: Runtime → Change runtime type → T4 GPU.*

# Notebook 03b — Foundation-Model Segmentation (Lab 3, Option B)

**Lab time.** 60 minutes.
**Tool.** Segment Anything (SAM) and the microscopy-tuned variant (μSAM).

**Learning goals.**

1. Describe the foundation-model paradigm and how it differs from task-specific models.
2. Apply SAM with point and box prompts to segment objects.
3. Compare foundation-model output to a Cellpose baseline.
4. Identify validation challenges specific to foundation-model output.
5. Decide when foundation-model approaches are preferable to task-specific models.

**Setup note.** SAM model weights are large (~360 MB for vit_b). On Colab T4 this downloads in ~30 seconds; on CPU expect slower inference. For runtime safety, this notebook can also run with `simulate_sam=True` to demonstrate the workflow without a real SAM download.

> **A note on the form widgets.** Several cells below use `#@param` comments. In **Google Colab** these render as interactive form widgets (sliders, dropdowns) at the top of the cell. In **other environments** (JupyterLab, VS Code, the JB rendered HTML) they appear as plain Python comments — edit the values directly and re-run the cell.

## Setup

In [ ]:
# SAM and helpers. micro-sam is large; we install it optionally.
%pip install --quiet "git+https://github.com/facebookresearch/segment-anything.git" matplotlib scikit-image numpy
import sys, os, urllib.request
import numpy as np
import matplotlib.pyplot as plt
from skimage import io as skio

IN_COLAB = "google.colab" in sys.modules
print("In Colab:", IN_COLAB)

In [ ]:
# We'll demo with simulated SAM behavior if downloading the real model fails.
# Toggle this to True to skip the model download.
SIMULATE_SAM = False
SAM_CKPT = "sam_vit_b_01ec64.pth"
SAM_URL  = "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth"

if not SIMULATE_SAM and not os.path.exists(SAM_CKPT):
    try:
        print(f"Downloading SAM model (~360 MB)...")
        urllib.request.urlretrieve(SAM_URL, SAM_CKPT)
        print(f"  Got {os.path.getsize(SAM_CKPT)//1024//1024} MB")
    except Exception as e:
        print(f"Download failed: {e}")
        print("Falling back to simulated SAM behavior so the lab still runs.")
        SIMULATE_SAM = True

## Load a non-canonical microscopy image

We use a synthetic image deliberately designed to fall outside Cellpose's training distribution: irregular shapes that look more like tissue than cultured cells.

In [ ]:
rng = np.random.default_rng(7)
size = 256

img = np.zeros((size, size), dtype=float)
# Add irregular blob-like structures
for _ in range(5):
    cy, cx = rng.integers(40, size-40, size=2)
    rr = rng.integers(15, 30)
    Y, X = np.ogrid[:size, :size]
    blob = ((Y-cy)/(rr*1.4))**2 + ((X-cx)/(rr*0.7))**2 <= 1
    img[blob] = rng.uniform(0.6, 1.0)
img = img + rng.normal(0, 0.05, img.shape)
img = np.clip(img, 0, 1)

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(img, cmap='gray'); ax.set_title("Non-canonical image"); ax.axis('off')
plt.tight_layout(); plt.show()

## Set up the SAM predictor

In [ ]:
if not SIMULATE_SAM:
    from segment_anything import sam_model_registry, SamPredictor
    sam = sam_model_registry["vit_b"](checkpoint=SAM_CKPT)
    predictor = SamPredictor(sam)
    # SAM expects a 3-channel uint8 image
    img_rgb = np.stack([img] * 3, axis=-1)
    img_rgb = (img_rgb * 255).astype(np.uint8)
    predictor.set_image(img_rgb)
    print("SAM ready.")
else:
    print("SIMULATE mode: skipping real SAM. We'll mock the prediction API.")

## Apply SAM with a point prompt

**What's happening here.** SAM takes one or more *prompts* — typically a foreground point, a background point, or a bounding box — and returns a segmentation mask of the object indicated. Unlike Cellpose, SAM does not produce a labeled image of all cells; it produces *one mask per prompt*. The model's job is "given this hint, what object am I being asked about?"

**Predict before running.** We're going to click a point inside one of the irregular blobs in the image. SAM was trained on natural images (mostly photographs) at large scale, then released with weights as a foundation model. What do you expect for our microscopy-style image?

- (a) SAM will produce a clean, tight boundary around the blob — better than Cellpose because it's a foundation model.
- (b) SAM will produce a rough boundary that covers the blob but extends into background — its priors don't quite match our image.
- (c) SAM will fail badly because the image is grayscale microscopy — the natural-image prior is the wrong distribution.
- (d) SAM will return *multiple candidate masks* and you need to choose — the tightest one will be reasonable, the loosest one will be terrible.

(d) is closest to truth. SAM's `multimask_output=True` returns three candidate masks at different scales (object / part / sub-part); the model also returns confidence scores so you can pick. The naive "trust SAM blindly" workflow misses this entirely.

In [ ]:
# Pick a point inside one of the blobs by inspecting the image
# (In real use, you'd click on it interactively in napari or similar.)
nonzero = np.argwhere(img > 0.5)
if len(nonzero) > 0:
    point = nonzero[len(nonzero) // 2]  # roughly middle of the brightest region
else:
    point = np.array([size // 2, size // 2])

input_point = np.array([[point[1], point[0]]])  # SAM uses (x, y) order
input_label = np.array([1])  # 1 = foreground

if not SIMULATE_SAM:
    masks, scores, _ = predictor.predict(
        point_coords=input_point,
        point_labels=input_label,
        multimask_output=True,
    )
    best = int(np.argmax(scores))
    sam_mask = masks[best]
    print(f"SAM returned {len(masks)} candidate masks; best score = {scores[best]:.3f}")
else:
    # Simulated: threshold around the prompt point
    Y, X = np.ogrid[:size, :size]
    sam_mask = (Y - point[0])**2 + (X - point[1])**2 <= 25**2

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(img, cmap='gray')
axes[0].plot(input_point[0, 0], input_point[0, 1], 'r*', markersize=18)
axes[0].set_title("Image + point prompt"); axes[0].axis('off')
axes[1].imshow(img, cmap='gray')
axes[1].imshow(np.where(sam_mask, 1, np.nan), cmap='autumn', alpha=0.5)
axes[1].set_title("SAM segmentation"); axes[1].axis('off')
plt.tight_layout(); plt.show()

**What you should be seeing.** SAM produced a mask around the chosen blob. The boundary is reasonable but typically not as tight as a task-trained model would give — answer (b) or (d) was right. If you ran with `SIMULATE_SAM=True`, the simulator returns a circular mask around the point regardless of the underlying image content; the real lesson requires the genuine model.

**Free exploration — drag the prompt point.** The cell below exposes the point coordinates as sliders. Try landing the point on different blobs, on the background, and right at a blob boundary. Note which prompt locations produce sensible masks and which produce the "object / part / sub-part" candidates that don't match what you intended.

In [ ]:
# @title Point-prompt exploration { run: "auto" }
prompt_x = 128  # @param {type: "slider", min: 0, max: 255, step: 4}
prompt_y = 128  # @param {type: "slider", min: 0, max: 255, step: 4}
multimask = True  # @param {type: "boolean"}

input_pt = np.array([[prompt_x, prompt_y]])
input_lbl = np.array([1])

if not SIMULATE_SAM:
    masks_e, scores_e, _ = predictor.predict(
        point_coords=input_pt,
        point_labels=input_lbl,
        multimask_output=multimask,
    )
    if multimask:
        # Show all 3 candidate masks side by side
        n = len(masks_e)
        fig, axes = plt.subplots(1, n, figsize=(4*n, 4))
        if n == 1:
            axes = [axes]
        for i, (mk, sc) in enumerate(zip(masks_e, scores_e)):
            axes[i].imshow(img, cmap='gray')
            axes[i].imshow(np.where(mk, 1, np.nan), cmap='autumn', alpha=0.5)
            axes[i].plot(prompt_x, prompt_y, 'r*', markersize=18)
            axes[i].set_title(f"Candidate {i}: score={sc:.3f}"); axes[i].axis('off')
    else:
        best_e = int(np.argmax(scores_e))
        fig, ax = plt.subplots(figsize=(5, 5))
        ax.imshow(img, cmap='gray')
        ax.imshow(np.where(masks_e[best_e], 1, np.nan), cmap='autumn', alpha=0.5)
        ax.plot(prompt_x, prompt_y, 'r*', markersize=18)
        ax.set_title(f"Best mask: score={scores_e[best_e]:.3f}"); ax.axis('off')
else:
    Y, X = np.ogrid[:size, :size]
    sim_mask = (Y - prompt_y)**2 + (X - prompt_x)**2 <= 25**2
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(img, cmap='gray')
    ax.imshow(np.where(sim_mask, 1, np.nan), cmap='autumn', alpha=0.5)
    ax.plot(prompt_x, prompt_y, 'r*', markersize=18)
    ax.set_title("Simulated mask (real SAM not loaded)"); ax.axis('off')
plt.tight_layout(); plt.show()

## Apply SAM with a bounding box prompt

In [ ]:
# Bounding box around (roughly) the same blob
y0, x0 = max(point[0] - 30, 0), max(point[1] - 30, 0)
y1, x1 = min(point[0] + 30, size), min(point[1] + 30, size)
input_box = np.array([x0, y0, x1, y1])

if not SIMULATE_SAM:
    masks_box, scores_box, _ = predictor.predict(
        box=input_box[None, :],
        multimask_output=False,
    )
    sam_box_mask = masks_box[0]
else:
    Y, X = np.ogrid[:size, :size]
    sam_box_mask = (Y >= y0) & (Y < y1) & (X >= x0) & (X < x1) & (img > 0.4)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(img, cmap='gray')
import matplotlib.patches as mpatches
rect = mpatches.Rectangle((x0, y0), x1-x0, y1-y0, linewidth=2, edgecolor='red', facecolor='none')
axes[0].add_patch(rect); axes[0].set_title("Image + box prompt"); axes[0].axis('off')
axes[1].imshow(img, cmap='gray')
axes[1].imshow(np.where(sam_box_mask, 1, np.nan), cmap='autumn', alpha=0.5)
axes[1].set_title("SAM segmentation (box)"); axes[1].axis('off')
plt.tight_layout(); plt.show()

**Compare the two outputs.** Did the point and box prompts produce similar masks? In our simple example they should; on real data they often diverge in informative ways. **The box prompt is generally more reliable** for objects with well-defined extents — you're telling SAM both "this object exists" and "it lives in this region." The point prompt is more flexible but more sensitive to where exactly you click.

## Prompt sensitivity test

**What's happening here.** Foundation models behave differently from task-specific models when you change the prompt. We'll move the point prompt around (left, center, right) and see how the mask responds.

**Predict before running.** As we shift the prompt point along the same blob, what do you expect SAM to do?

- (a) Return identical masks — the model recognizes the blob and the exact prompt location doesn't matter.
- (b) Return slightly different masks — boundary jitter at the level of a few pixels.
- (c) Return *qualitatively* different masks — sometimes the whole blob, sometimes a sub-part, sometimes the wrong object entirely.
- (d) Crash if the prompt isn't on a blob.

(c) is closest to the lesson. Foundation models are *prompt-sensitive* in ways that task-specific models are not. The same biological object can yield three different masks depending on where you click — and the user has no way to tell *which* mask is "correct" without ground truth. This is the central validation challenge of foundation models.

In [ ]:
# Move the point around and see how the mask changes
shifts = [(-20, 0), (0, 0), (20, 0)]
fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
for i, (dy, dx) in enumerate(shifts):
    test_point = point + np.array([dy, dx])
    test_input = np.array([[test_point[1], test_point[0]]])

    if not SIMULATE_SAM:
        masks_t, scores_t, _ = predictor.predict(
            point_coords=test_input,
            point_labels=np.array([1]),
            multimask_output=True,
        )
        best_t = int(np.argmax(scores_t))
        sam_mask_t = masks_t[best_t]
    else:
        Y, X = np.ogrid[:size, :size]
        sam_mask_t = (Y - test_point[0])**2 + (X - test_point[1])**2 <= 25**2

    axes[i].imshow(img, cmap='gray')
    axes[i].imshow(np.where(sam_mask_t, 1, np.nan), cmap='autumn', alpha=0.5)
    axes[i].plot(test_input[0, 0], test_input[0, 1], 'r*', markersize=15)
    axes[i].set_title(f"Point shifted by ({dy}, {dx})")
    axes[i].axis('off')
plt.tight_layout(); plt.show()

**What you should be seeing.** Three slight prompt shifts produced three meaningfully different masks. Sometimes the difference is a few boundary pixels; sometimes the model returned the whole object versus a sub-part of it. **This is foundation-model behavior in a nutshell.** Task-specific models are deterministic for a given input; foundation models are deterministic only for a given (input, prompt) pair, and changing the prompt is the user's responsibility.

**The validation challenge with foundation models.** Output depends on the prompt. There is no fixed 'training distribution' to validate against the way you would for a task-specific model.

For Lab 3b's setting, the right validation evidence asks:

- *Reproducibility* — does the same prompt always give the same output? (Yes, deterministic.)
- *Prompt sensitivity* — how much does the output change as the prompt moves? (Often: surprisingly little, but sometimes: a lot.)
- *Coverage* — for an automatic-mask-generation pass, does it produce all the objects you expect? Are there false positives?

## When to use foundation models vs task-specific models

In [ ]:
decision_framework = '''
Use task-specific pretrained models (Cellpose, Stardist, Mesmer) when:
  - The model's training distribution covers your data
  - The model has been validated on cases similar to yours
  - You need consistent, automated batch processing

Use foundation models with prompts (SAM, μSAM) when:
  - No fitting task-specific model exists
  - You have a small number of images and can prompt per-image
  - You're doing exploratory analysis on novel sample types
  - You want flexible, prompt-driven segmentation

Train a custom model (fine-tune Cellpose, fine-tune a foundation model) when:
  - You have labeled data
  - The volume of work justifies the investment
  - The project will run for months or years
'''
print(decision_framework)

## Going broader — beyond SAM and μSAM

Notebook 04 catalogs the broader foundation-model and community-platform ecosystem:

- **BioImage Model Zoo browser** — live API to list, filter, and load *hundreds* of pretrained models. The "choose on the fly" pattern: at workshop time, browse what's available for your task and load it inline. *Notebook 04, Section 2.*
- **fnet (label-free prediction)** — predicts fluorescence from brightfield without staining. Demonstrates the generation-task pattern. *Notebook 04, inline demo.*
- **pix2pix (image-to-image translation)** — supervised translation between paired image domains. Common bioimage uses: virtual staining, modality translation. *Notebook 04, inline demo.*
- **CycleGAN** — unpaired translation, no paired data needed. *Notebook 04, catalog.*

Foundation models extend reach; the community catalog extends the menu of alternatives. Start with BiMZ when prototyping; if a pretrained model exists for your task, you save the training cost.

## Closing reflection

Foundation-model segmentation extends reach into novel sample types at the cost of greater prompt-engineering and validation work. The choice between task-specific and foundation-model approaches is a deliberate trade-off, not a default.

**Where to go next:**

- The [μSAM (micro-sam) GitHub repository](https://github.com/computational-cell-analytics/micro-sam) for the microscopy-specific SAM variant.
- [napari plugins for SAM](https://www.napari-hub.org/?search=sam) for interactive prompting.
- The [Cellpose-SAM](https://github.com/MouseLand/cellpose) v4 release combines task-specific Cellpose with the SAM transformer backbone — worth comparing.